In [48]:
from nba_api.stats.static import teams
from nba_api.stats.endpoints import commonteamroster, playercareerstats

In [28]:
lakers = teams.find_teams_by_full_name('los angeles lakers')
lakers_id = lakers[0]['id']
celtics = teams.find_teams_by_full_name('boston celtics')
celtics_id = celtics[0]['id']
knicks = teams.find_teams_by_full_name('new york knicks')
knicks_id = knicks[0]['id']

In [69]:
lakers_roster = commonteamroster.CommonTeamRoster(team_id=lakers_id).get_data_frames()[0]
celtics_roster = commonteamroster.CommonTeamRoster(team_id=celtics_id).get_data_frames()[0]
knicks_roster = commonteamroster.CommonTeamRoster(team_id=knicks_id).get_data_frames()[0]

In [93]:
import pandas as pd
import json

def transform_to_dynamo_json(df, details):
    # Create a list of Maps. Each map has 'name' (String) and 'player_id' (Number)
    player_list = [
        {
            "M": {
                "name": {"S": row['PLAYER']},
                "player_id": {"N": str(row['PLAYER_ID'])}
            }
        } 
        for _, row in df.iterrows()
    ]
    
    print(f"Processed {len(player_list)} players for {details['full_name']}")

    dynamo_item = {
        "team_id": {
            "N": str(df['TeamID'].iloc[0]) # Using iloc for cleaner access
        },
        "name": {
            "S": details['full_name']
        },
        "team": {
            "M": {
                "players": {
                    "L": player_list
                }
            }
        }
    }
    
    return dynamo_item

# Your existing logic to save the files
teams_to_process = [
    (lakers_roster, lakers[0], 'lakers'),
    (celtics_roster, celtics[0], 'celtics'),
    (knicks_roster, knicks[0], 'knicks')
]

for roster_df, team_details, slug in teams_to_process:
    formatted_data = transform_to_dynamo_json(roster_df, team_details)
    with open(f'player_data/formatted_data_{slug}.json', 'w') as f:
        json.dump(formatted_data, f, indent=4)

Processed 18 players for Los Angeles Lakers
Processed 15 players for Boston Celtics
Processed 18 players for New York Knicks


In [94]:
import os
import json

player_data_dir = 'player_data'
# Filter specifically for the 'formatted_data_' files we just created
player_files = [f for f in os.listdir(player_data_dir) if f.startswith('formatted_data_') and f.endswith('.json')]

all_player_data = {}

for file in player_files:
    with open(os.path.join(player_data_dir, file), 'r') as f:
        data = json.load(f)
        
        # Extract the ID and the player list
        t_id = data['team_id']['N']
        
        # UPDATED PROCESSING: 
        # We now dig into the Map 'M' to get the 'player_id'
        players_in_file = [
            player['M']['player_id']['N'] 
            for player in data['team']['M']['players']['L']
        ]
        
        # Store in our dictionary
        if t_id in all_player_data:
            all_player_data[t_id].extend(players_in_file)
        else:
            all_player_data[t_id] = players_in_file

# Save the combined mapping for your next script
with open('combined_team_player_map.json', 'w') as f:
    json.dump(all_player_data, f, indent=4)

print(f"Combined data from {len(player_files)} teams. Total teams in map: {len(all_player_data)}")

Combined data from 3 teams. Total teams in map: 3


In [106]:
import time
import json
import os
from tqdm import tqdm
from nba_api.stats.endpoints import playercareerstats, commonplayerinfo

TABLE_NAME = 'Players'
batch_items = []
batch_count = 0

for team_id, player_ids in all_player_data.items():
    for player_id in tqdm(player_ids, desc=f"Processing Team {team_id}"):
        try:
            # 1. Fetch Career Stats (Shooting Data)
            career = playercareerstats.PlayerCareerStats(player_id=player_id)
            career_df = career.get_data_frames()[0]

            # 2. Fetch Player Bio Info (Biographical Data)
            player_info = commonplayerinfo.CommonPlayerInfo(player_id=player_id)
            info_df = player_info.get_data_frames()[0]

            if not career_df.empty and not info_df.empty:
                stats_row = career_df.iloc[-1] 
                bio_row = info_df.iloc[0]

                # Shooting Ingredients
                p3_1 = int(stats_row['FG3M'])
                p3_0 = int(stats_row['FG3A'] - stats_row['FG3M'])
                p2_1 = int(stats_row['FGM'] - stats_row['FG3M'])
                p2_0 = int((stats_row['FGA'] - stats_row['FG3A']) - (stats_row['FGM'] - stats_row['FG3M']))
                ft1 = int(stats_row['FTM'])
                ft0 = int(stats_row['FTA'] - stats_row['FTM'])

                # Constructing the Headshot URL
                headshot_url = f"https://cdn.nba.com/headshots/nba/latest/1040x760/{player_id}.png"

                # Constructing the Item with Bio, Headshot, and Nested Stats
                put_request = {
                    "PutRequest": {
                        "Item": {
                            "player_id": {"N": str(player_id)},
                            "team_id": {"N": str(team_id)},
                            "name": {"S": str(bio_row['DISPLAY_FIRST_LAST'])},
                            "headshot_url": {"S": headshot_url}, # NEW ATTRIBUTE
                            "birthdate": {"S": str(bio_row['BIRTHDATE'])},
                            "height": {"S": str(bio_row['HEIGHT'])},
                            "weight": {"S": str(bio_row['WEIGHT'])},
                            "jersey": {"S": str(bio_row['JERSEY'])},
                            "position": {"S": str(bio_row['POSITION'])},
                            "draft_year": {"S": str(bio_row['DRAFT_YEAR'])},
                            "stats": {
                                "M": {
                                    "p2_1": {"N": str(p2_1)},
                                    "p2_0": {"N": str(p2_0)},
                                    "p3_1": {"N": str(p3_1)},
                                    "p3_0": {"N": str(p3_0)},
                                    "ft1": {"N": str(ft1)},
                                    "ft0": {"N": str(ft0)},
                                    "total_points": {"N": str(stats_row['PTS'])}
                                }
                            }
                        }
                    }
                }
                
                batch_items.append(put_request)

            # Batching logic: Save every 25 items
            if len(batch_items) == 25:
                output_path = f'player_data/batch_{batch_count}.json'
                os.makedirs(os.path.dirname(output_path), exist_ok=True)
                with open(output_path, 'w') as f:
                    json.dump({TABLE_NAME: batch_items}, f, indent=4)
                
                batch_items = []
                batch_count += 1

            time.sleep(0.8) 
        except Exception as e:
            print(f" Error for {player_id}: {e}")
            continue

# Final catch for remaining items
if batch_items:
    output_path = f'player_data/batch_{batch_count}.json'
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    with open(output_path, 'w') as f:
        json.dump({TABLE_NAME: batch_items}, f, indent=4)

print(f"\nDone! Created batch files with headshot URLs.")

Processing Team 1610612752: 100%|██████████| 18/18 [00:22<00:00,  1.26s/it]


Done! Created batch files with headshot URLs.


In [107]:
from nba_api.stats.endpoints import commonplayerinfo
wiggs = commonplayerinfo.CommonPlayerInfo(203952)
wiggs.get_data_frames()[0]

# render the image from the url
url = "https://cdn.nba.com/headshots/nba/latest/1040x760/2544.png"

# render the image from the url
from IPython.display import Image
# Image(url)